# Patient-level split and genomic preprocessing

**EXSU 500 — Prostate cancer stage classification**

This notebook performs:
- Reproducible 70/15/15 patient-level splitting
- CNA feature matrix construction
- Mutation sequencing-panel coverage validation

Mutation feature encoding, imputation, feature selection,
and model fitting will be completed in subsequent steps.

In [1]:
import json
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

DATA_DIR = Path("../data/processed")
PANEL_DIR = Path("../data/raw/panels")
RANDOM_STATE = 42

## 1. Patient-level splitting

Patients are randomly divided into training (70%), validation (15%),
and held-out test (15%) sets, stratified by cancer stage.

Splitting by patient ID prevents patient overlap between datasets.
The test set is reserved for final model evaluation.

In [2]:
clinical = pd.read_csv(DATA_DIR / "clinical_manifest.csv")

required_cols = {"PATIENT_ID", "SAMPLE_ID", "stage_binary"}

if not required_cols.issubset(clinical.columns):
    raise ValueError(
        f"Missing clinical columns: {required_cols - set(clinical.columns)}"
    )

if clinical[list(required_cols)].isna().any().any():
    raise ValueError("Missing patient, sample, or stage identifiers.")

if not clinical["stage_binary"].isin([0, 1]).all():
    raise ValueError("Stage labels must be binary (0 or 1).")

if clinical.groupby("PATIENT_ID")["stage_binary"].nunique().gt(1).any():
    raise ValueError("Conflicting stage labels for the same patient.")

# One row per patient
patients = clinical[
    ["PATIENT_ID", "stage_binary"]
].drop_duplicates("PATIENT_ID")

# 70% training, 30% remaining
train_patients, remaining = train_test_split(
    patients,
    test_size=0.30,
    stratify=patients["stage_binary"],
    random_state=RANDOM_STATE,
)

# Split remaining patients equally
val_patients, test_patients = train_test_split(
    remaining,
    test_size=0.50,
    stratify=remaining["stage_binary"],
    random_state=RANDOM_STATE,
)

# Combine patient assignments
splits = pd.concat(
    [
        train_patients.assign(split="train"),
        val_patients.assign(split="validation"),
        test_patients.assign(split="test"),
    ],
    ignore_index=True,
)

# Validate split integrity
assert splits["PATIENT_ID"].is_unique
assert len(splits) == patients["PATIENT_ID"].nunique()
assert set(splits["PATIENT_ID"]) == set(patients["PATIENT_ID"])

# Save assignments
splits.to_csv(DATA_DIR / "patient_splits.csv", index=False)

print("Patients per split and stage:")
print(pd.crosstab(
    splits["split"],
    splits["stage_binary"],
    margins=True
))

print("PASS: Unique patient assignments; saved patient_splits.csv")

Patients per split and stage:
stage_binary    0     1   All
split                        
test          142   154   296
train         660   716  1376
validation    141   154   295
All           943  1024  1967
PASS: Unique patient assignments; saved patient_splits.csv


## 2. CNA feature matrix

Convert long-format CNA records into a patient-by-gene matrix.

Original CNA values are preserved, including -1.5.
Feature filtering and transformations are deferred until modelling.

In [3]:
cna = pd.read_csv(DATA_DIR / "cna_long.csv")

# Verify unique sample-gene pairs
if cna.duplicated(["SAMPLE_ID", "HUGO_SYMBOL"]).any():
    raise ValueError("Duplicate CNA sample-gene pairs.")

if clinical["SAMPLE_ID"].duplicated().any():
    raise ValueError("Duplicate sample IDs in clinical manifest.")

# Create sample-by-gene matrix
X_cna = cna.pivot(
    index="SAMPLE_ID",
    columns="HUGO_SYMBOL",
    values="CNA_VALUE"
).reindex(clinical["SAMPLE_ID"].tolist())

# Rename features and align patients
X_cna.columns = [f"CNA__{gene}" for gene in X_cna.columns]
X_cna.index = pd.Index(
    clinical["PATIENT_ID"],
    name="PATIENT_ID"
)

# Validate
assert X_cna.index.is_unique
assert not X_cna.isna().any().any()

print(f"CNA matrix: {X_cna.shape}")
print(
    "CNA categories:",
    sorted(pd.unique(X_cna.to_numpy().ravel()).tolist())
)

# Save unfiltered CNA matrix
X_cna.to_csv(DATA_DIR / "cna_feature_matrix.csv")

print("Saved cna_feature_matrix.csv")

CNA matrix: (1967, 541)
CNA categories: [-2.0, -1.5, 0.0, 2.0]
Saved cna_feature_matrix.csv


## 3. Mutation sequencing-panel coverage audit

Patients were sequenced using different MSK-IMPACT panels.

Mutation records are checked against their assigned panel's gene list.
Unmatched records are identified but not removed in this notebook.

Mutation encoding will be completed after resolving or explicitly
excluding unmatched records.

In [4]:
mutations = pd.read_csv(DATA_DIR / "mutations_long.csv")
panels = pd.read_csv(DATA_DIR / "gene_panel_matrix.csv")

panel_ids = [
    "IMPACT341",
    "IMPACT410",
    "IMPACT468",
    "IMPACT505"
]

# Load panel gene lists
panel_genes = {}

for panel_id in panel_ids:
    with open(
        PANEL_DIR / f"{panel_id}.json",
        encoding="utf-8-sig"
    ) as f:
        panel = json.load(f)

    panel_genes[panel_id] = {
        gene["hugoGeneSymbol"]
        for gene in panel["genes"]
    }

# Assign panel IDs to mutation records
sample_panels = panels[
    ["SAMPLE_ID", "mutations"]
].rename(columns={"mutations": "PANEL_ID"})

mutation_audit = mutations.merge(
    sample_panels,
    on="SAMPLE_ID",
    how="left",
    validate="many_to_one"
)

if mutation_audit["PANEL_ID"].isna().any():
    raise ValueError("Missing panel assignments.")

if not mutation_audit["PANEL_ID"].isin(panel_genes).all():
    raise ValueError("Unknown sequencing-panel ID.")

# Check panel coverage
mutation_audit["covered"] = [
    gene in panel_genes[panel]
    for gene, panel in zip(
        mutation_audit["HUGO_SYMBOL"],
        mutation_audit["PANEL_ID"]
    )
]

unmatched = mutation_audit.loc[
    ~mutation_audit["covered"]
]

print(
    "Panel gene counts:",
    {k: len(v) for k, v in panel_genes.items()}
)

print(f"Total mutations: {len(mutation_audit):,}")
print(f"Matching mutations: {mutation_audit['covered'].sum():,}")
print(f"Unmatched mutations: {len(unmatched):,}")

print("\nUnmatched mutations by panel:")
print(unmatched["PANEL_ID"].value_counts())

Panel gene counts: {'IMPACT341': 341, 'IMPACT410': 410, 'IMPACT468': 469, 'IMPACT505': 505}
Total mutations: 9,651
Matching mutations: 9,440
Unmatched mutations: 211

Unmatched mutations by panel:
PANEL_ID
IMPACT468    158
IMPACT410     26
IMPACT505     20
IMPACT341      7
Name: count, dtype: int64


## 4. Next steps

The following preprocessing tasks remain:

1. Exclude or reconcile 211 unmatched mutation records.
2. Build the coverage-aware mutation feature matrix.
3. Prepare mutation-only, CNA-only, and combined feature sets.
4. Apply training-only feature filtering and missing-value handling.
5. Train logistic regression and random forest models.

### Limitations

The stage label represents the highest recorded cancer stage,
which may differ from the stage at specimen collection.

Different sequencing panels cover different gene sets.
Uncovered genes must not be automatically treated as mutation-negative.